In [4]:
# ============================================================
# PREPROCESSING FUER HAZMAT-VRP (FINAL)
# ============================================================

import pandas as pd
import numpy as np
import pickle
import networkx as nx

# ============================================================
# 1. EINSTELLUNGEN
# ============================================================

ACCIDENTS_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_accidents.pkl"
POPULATION_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_population.pkl"
NATURE_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_nature.pkl"

OUTPUT_MATRIX_CSV = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\matrix.csv"

BOUNDING_BUFFER_DEG = 5.0   # 🔥 WICHTIG: größerer Graph
AVG_SPEED_KMH = 70.0

PATH_PROFILES = {
    "shortest": 0.0,
    "balanced": 1.0,
    "safest": 5.0   # 🔥 stärkerer Risiko-Fokus
}

alpha, beta, gamma = 0.4, 0.4, 0.2

# ============================================================
# 2. HILFSFUNKTIONEN
# ============================================================

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1 = map(np.radians, [lat1, lon1])
    lat2, lon2 = map(np.radians, [lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def normalize_dict(d, invert=False):
    if not d:
        return {}
    vals = np.array(list(d.values()))
    vmin, vmax = vals.min(), vals.max()
    if vmin == vmax:
        return {k: 0.0 for k in d}
    out = {}
    for k, v in d.items():
        x = (v - vmin) / (vmax - vmin + 1e-9)
        out[k] = 1 - x if invert else x
    return out

# ============================================================
# 3. RELEVANTE PUNKTE
# ============================================================

points_df = pd.DataFrame([
    {"id": "DEPOT", "type": "depot", "lat": 52.52, "lon": 13.405},
    {"id": "C1", "type": "customer", "lat": 48.1374, "lon": 11.5755},
    {"id": "C2", "type": "customer", "lat": 50.1109, "lon": 8.6821},
    {"id": "C3", "type": "customer", "lat": 53.5511, "lon": 9.9937},
    {"id": "C4", "type": "customer", "lat": 51.2277, "lon": 6.7735},
])

lat_min = points_df.lat.min() - BOUNDING_BUFFER_DEG
lat_max = points_df.lat.max() + BOUNDING_BUFFER_DEG
lon_min = points_df.lon.min() - BOUNDING_BUFFER_DEG
lon_max = points_df.lon.max() + BOUNDING_BUFFER_DEG

# ============================================================
# 4. GRAPH LADEN + REDUZIEREN
# ============================================================

with open(ACCIDENTS_PATH, "rb") as f:
    g = pickle.load(f)

nodes = g["nodes"]
edges = g["edges"]

nodes = nodes[
    (nodes.lat >= lat_min) &
    (nodes.lat <= lat_max) &
    (nodes.lon >= lon_min) &
    (nodes.lon <= lon_max)
].copy()

valid_nodes = set(nodes.node)

edges = edges[
    edges.u.isin(valid_nodes) &
    edges.v.isin(valid_nodes)
].copy()

print(f"Nodes: {len(nodes)} | Edges: {len(edges)}")

# ============================================================
# 5. RISIKO LADEN
# ============================================================

def load_metric(path, col):
    with open(path, "rb") as f:
        g = pickle.load(f)
    df = g["edges"]
    df = df[df.u.isin(valid_nodes) & df.v.isin(valid_nodes)]
    return dict(zip(zip(df.u, df.v), df[col]))

pop = normalize_dict(load_metric(POPULATION_PATH, "pop_per_meter"))
acc = normalize_dict(load_metric(ACCIDENTS_PATH, "score"))
nat = normalize_dict(load_metric(NATURE_PATH, "dist_to_nature_m"), invert=True)

# ============================================================
# 6. GRAPH AUFBAUEN (UNDIRECTED)
# ============================================================

G = nx.Graph()

for _, row in edges.iterrows():
    u, v = int(row.u), int(row.v)

    dist = float(row.length) / 1000.0  # km

    # 🔥 WICHTIG: Risiko richtig skaliert
    risk_val = beta * acc.get((u,v),0) * (
        alpha * pop.get((u,v),0) + gamma * nat.get((u,v),0)
    )

    risk_val *= 1000   # 🔥 entscheidend!

    G.add_edge(u, v, dist=dist, risk=risk_val)

print("✅ Graph gebaut")

# ============================================================
# 7. NODE MAPPING
# ============================================================

node_ids = nodes.node.values
node_lats = nodes.lat.values
node_lons = nodes.lon.values

def nearest(lat, lon):
    d = haversine_vectorized(lat, lon, node_lats, node_lons)
    return int(node_ids[np.argmin(d)])

points_df["node"] = [
    nearest(r.lat, r.lon) for _, r in points_df.iterrows()
]

print(points_df)

# ============================================================
# 8. DIJKSTRA
# ============================================================

def dijkstra_map(source, lam):
    def weight(u, v, d):
        return d["dist"] + lam * d["risk"]
    return nx.single_source_dijkstra_path_length(G, source, weight=weight)

# ============================================================
# 9. OD MATRIX
# ============================================================

records = []

for _, src in points_df.iterrows():

    dist_only = nx.single_source_dijkstra_path_length(
        G, src.node, weight=lambda u,v,d: d["dist"]
    )

    risk_only = nx.single_source_dijkstra_path_length(
        G, src.node, weight=lambda u,v,d: d["risk"]
    )

    for profile, lam in PATH_PROFILES.items():

        total_map = dijkstra_map(src.node, lam)

        for _, dst in points_df.iterrows():

            if src.id == dst.id:
                continue

            if dst.node not in total_map:
                continue

            dist_km = dist_only.get(dst.node, np.nan)
            risk = risk_only.get(dst.node, np.nan)

            time_min = 60 * dist_km / AVG_SPEED_KMH

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "dist_km": dist_km,
                "time_min": time_min,
                "risk": risk
            })

# ============================================================
# 10. SPEICHERN
# ============================================================

od_df = pd.DataFrame(records)

# 🔥 zur Kontrolle direkt anzeigen
print(od_df.head(20))

od_df.to_csv(OUTPUT_MATRIX_CSV, index=False)

print("✅ OD Matrix fertig")

# ============================================================
# 11. REALITY CHECK
# ============================================================

for _, row in od_df.iterrows():
    if row["from"] == "DEPOT" and row["to"] == "C1":

        luftlinie = haversine_vectorized(
            52.52, 13.405,
            48.1374, 11.5755
        ) / 1000

        print("\n--- CHECK BERLIN -> MÜNCHEN ---")
        print("Modell:", row["dist_km"])
        print("Luftlinie:", luftlinie)
        print("Ratio:", row["dist_km"] / luftlinie)

Nodes: 2020869 | Edges: 3025675
✅ Graph gebaut
      id      type      lat      lon        node
0  DEPOT     depot  52.5200  13.4050  1827996421
1     C1  customer  48.1374  11.5755    27322865
2     C2  customer  50.1109   8.6821      651510
3     C3  customer  53.5511   9.9937  6694197251
4     C4  customer  51.2277   6.7735   283331394
     from     to   profile      dist_km    time_min        risk
0   DEPOT     C1  shortest   912.225806  781.907834   96.236467
1   DEPOT     C2  shortest   833.300070  714.257203  115.505620
2   DEPOT     C3  shortest   477.389173  409.190719   82.822933
3   DEPOT     C4  shortest   899.207260  770.749080  106.552554
4   DEPOT     C1  balanced   912.225806  781.907834   96.236467
5   DEPOT     C2  balanced   833.300070  714.257203  115.505620
6   DEPOT     C3  balanced   477.389173  409.190719   82.822933
7   DEPOT     C4  balanced   899.207260  770.749080  106.552554
8   DEPOT     C1    safest   912.225806  781.907834   96.236467
9   DEPOT     C2   

In [ ]:
ACCIDENTS_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_accidents.pkl"
POPULATION_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_population.pkl"
NATURE_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_nature.pkl"

In [6]:
import pickle
import pandas as pd
import numpy as np
import networkx as nx

# ============================================
# 1. DATEN LADEN UND MERGEN
# ============================================

ACCIDENTS_PATH = "germany_graph_with_accidents.pkl"
POPULATION_PATH = "germany_graph_with_population.pkl"

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_accidents.pkl", "rb") as f:
    acc_data = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_population.pkl", "rb") as f:
    pop_data = pickle.load(f)

edges = acc_data["edges"].copy()
nodes = acc_data["nodes"].copy()

# Pop-Score hinzufügen
pop_map = dict(zip(
    zip(pop_data["edges"]["u"], pop_data["edges"]["v"]),
    pop_data["edges"]["pop_per_meter"]
))
edges["pop_risk"] = edges.apply(
    lambda r: pop_map.get((r["u"], r["v"]), 0), axis=1
)

print(f"Kanten: {len(edges)}, Knoten: {len(nodes)}")

# ============================================
# 2. EINHEITEN PRÜFEN
# ============================================

print("\n--- Distanz-Statistik ---")
print(edges["distance"].describe())

# Automatische Erkennung
median_dist = edges["distance"].median()
if median_dist > 500:
    DIST_DIVISOR = 1000.0  # Meter → km
    print("→ Distanzen in Metern erkannt")
else:
    DIST_DIVISOR = 1.0     # bereits km
    print("→ Distanzen bereits in km")

# ============================================
# 3. GRAPH AUFBAUEN
# ============================================

G = nx.DiGraph()

for _, row in edges.iterrows():
    u, v = int(row["u"]), int(row["v"])
    
    dist_km = float(row["distance"]) / DIST_DIVISOR
    
    # Normalisierter Risiko-Score (0-1 Bereich)
    acc_risk = float(row.get("score", 0))
    pop_risk = float(row.get("pop_risk", 0))
    
    G.add_edge(u, v, dist=dist_km, risk=acc_risk + pop_risk)

print(f"\nGraph: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten")

# ============================================
# 4. KONNEKTIVITÄT
# ============================================

if not nx.is_weakly_connected(G):
    components = list(nx.weakly_connected_components(G))
    print(f"⚠️ {len(components)} unverbundene Komponenten!")
    
    largest = max(components, key=len)
    G = G.subgraph(largest).copy()
    print(f"→ Reduziert auf größte: {G.number_of_nodes()} Knoten")

# ============================================
# 5. NEAREST NODE FUNKTION
# ============================================

node_arr = nodes[["node", "lat", "lon"]].values
node_ids = node_arr[:, 0].astype(int)
node_lats = node_arr[:, 1].astype(float)
node_lons = node_arr[:, 2].astype(float)

def nearest_node(lat, lon):
    # Einfache euklidische Näherung (für Deutschland OK)
    d = np.sqrt((node_lats - lat)**2 + (node_lons - lon)**2)
    return int(node_ids[np.argmin(d)])

# ============================================
# 6. TEST
# ============================================

berlin = nearest_node(52.52, 13.405)
munich = nearest_node(48.1374, 11.5755)

print(f"\nBerlin Node: {berlin}")
print(f"München Node: {munich}")

try:
    dist = nx.shortest_path_length(G, berlin, munich, weight="dist")
    print(f"Kürzeste Distanz: {dist:.1f} km")
    
    if 500 < dist < 700:
        print("✓ Distanz plausibel!")
    else:
        print("⚠️ Distanz unplausibel — Einheiten prüfen!")
        
except nx.NetworkXNoPath:
    print("❌ Kein Pfad gefunden — Knoten in verschiedenen Komponenten!")


Kanten: 3025675, Knoten: 2020869

--- Distanz-Statistik ---
count    3.025675e+06
mean     3.847954e+01
std      5.596571e+01
min      6.827160e-03
25%      1.113236e+01
50%      2.214154e+01
75%      4.470371e+01
max      3.500079e+03
Name: distance, dtype: float64
→ Distanzen bereits in km

Graph: 2020869 Knoten, 3025663 Kanten
⚠️ 88 unverbundene Komponenten!
→ Reduziert auf größte: 2015787 Knoten


C:\Users\tspol\AppData\Local\Temp\ipykernel_12736\1197878083.py:85: RuntimeWarning: invalid value encountered in cast
  node_ids = node_arr[:, 0].astype(int)



Berlin Node: 1827996421
München Node: 1565811201
Kürzeste Distanz: 590683.8 km
⚠️ Distanz unplausibel — Einheiten prüfen!


In [2]:
od_df = pd.DataFrame(records)

In [3]:
print(od_df.head(20))

     from     to   profile      dist_km    time_min       risk
0   DEPOT     C1  shortest   912.225806  781.907834  48.118233
1   DEPOT     C2  shortest   833.300070  714.257203  57.752810
2   DEPOT     C3  shortest   477.389173  409.190719  41.411467
3   DEPOT     C4  shortest   899.207260  770.749080  53.276277
4   DEPOT     C1  balanced   912.225806  781.907834  48.118233
5   DEPOT     C2  balanced   833.300070  714.257203  57.752810
6   DEPOT     C3  balanced   477.389173  409.190719  41.411467
7   DEPOT     C4  balanced   899.207260  770.749080  53.276277
8   DEPOT     C1    safest   912.225806  781.907834  48.118233
9   DEPOT     C2    safest   833.300070  714.257203  57.752810
10  DEPOT     C3    safest   477.389173  409.190719  41.411467
11  DEPOT     C4    safest   899.207260  770.749080  53.276277
12     C1  DEPOT  shortest   912.225806  781.907834  48.118233
13     C1     C2  shortest   566.289038  485.390604  35.995663
14     C1     C3  shortest  1147.727797  983.766683  36

In [5]:
# ============================================================
# PREPROCESSING FUER HAZMAT-VRP (FINAL STABLE VERSION)
# ============================================================

import pandas as pd
import numpy as np
import pickle
import networkx as nx
import heapq

# ============================================================
# SETTINGS
# ============================================================

ACCIDENTS_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_accidents.pkl"
POPULATION_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_population.pkl"
NATURE_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_nature.pkl"

OUTPUT_MATRIX_CSV = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_od_matrix_final.csv"

BOUNDING_BUFFER_DEG = 1.5
AVG_SPEED_KMH = 70.0

PATH_PROFILES = {
    "shortest": 0.0,
    "balanced": 1.0,
    "safest": 4.0
}

alpha, beta, gamma = 0.4, 0.4, 0.2

# ============================================================
# HELPER
# ============================================================

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1 = map(np.radians, [lat1, lon1])
    lat2, lon2 = map(np.radians, [lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def normalize_dict(d, invert=False):
    if not d:
        return {}
    vals = np.array(list(d.values()))
    vmin, vmax = vals.min(), vals.max()
    if vmin == vmax:
        return {k: 0.0 for k in d}
    return {
        k: (1 - (v - vmin)/(vmax - vmin + 1e-9)) if invert
        else (v - vmin)/(vmax - vmin + 1e-9)
        for k, v in d.items()
    }

# ============================================================
# POINTS
# ============================================================

points_df = pd.DataFrame([
    {"id": "DEPOT", "lat": 52.52, "lon": 13.405},
    {"id": "C1", "lat": 48.1374, "lon": 11.5755},
    {"id": "C2", "lat": 50.1109, "lon": 8.6821},
    {"id": "C3", "lat": 53.5511, "lon": 9.9937},
    {"id": "C4", "lat": 51.2277, "lon": 6.7735},
])

lat_min = points_df.lat.min() - BOUNDING_BUFFER_DEG
lat_max = points_df.lat.max() + BOUNDING_BUFFER_DEG
lon_min = points_df.lon.min() - BOUNDING_BUFFER_DEG
lon_max = points_df.lon.max() + BOUNDING_BUFFER_DEG

# ============================================================
# LOAD GRAPH
# ============================================================

with open(ACCIDENTS_PATH, "rb") as f:
    g = pickle.load(f)

nodes = g["nodes"]
edges = g["edges"]

nodes = nodes[
    (nodes.lat >= lat_min) &
    (nodes.lat <= lat_max) &
    (nodes.lon >= lon_min) &
    (nodes.lon <= lon_max)
].copy()

valid_nodes = set(nodes.node)

edges = edges[
    edges.u.isin(valid_nodes) &
    edges.v.isin(valid_nodes)
].copy()

print(f"Nodes: {len(nodes)} | Edges: {len(edges)}")

# ============================================================
# OPTIONAL: HIGHWAY FILTER
# ============================================================

if "highway" in edges.columns:
    edges = edges[edges["highway"].isin([
        "motorway",
        "trunk",
        "primary",
        "secondary"
    ])]

# ============================================================
# LOAD RISKS
# ============================================================

def load_metric(path, col):
    with open(path, "rb") as f:
        g = pickle.load(f)
    df = g["edges"]
    df = df[df.u.isin(valid_nodes) & df.v.isin(valid_nodes)]
    return dict(zip(zip(df.u, df.v), df[col]))

pop = normalize_dict(load_metric(POPULATION_PATH, "pop_per_meter"))
acc = normalize_dict(load_metric(ACCIDENTS_PATH, "score"))
nat = normalize_dict(load_metric(NATURE_PATH, "dist_to_nature_m"), invert=True)

# ============================================================
# NODE COORDS
# ============================================================

node_coords = nodes.set_index("node")[["lat", "lon"]].to_dict("index")

# ============================================================
# BUILD GRAPH
# ============================================================

G = nx.DiGraph()

for _, row in edges.iterrows():
    u, v = int(row.u), int(row.v)

    # ✅ DISTANZ robust
    if "length" in row and 0 < row.length < 10000:
        dist = float(row.length) / 1000.0
    else:
        cu = node_coords[u]
        cv = node_coords[v]
        dist = haversine(cu["lat"], cu["lon"], cv["lat"], cv["lon"]) / 1000.0

    # ✅ RISIKO
    risk = beta * acc.get((u,v), 0) * (
        alpha * pop.get((u,v), 0) + gamma * nat.get((u,v), 0)
    ) * 500

    G.add_edge(u, v, dist=dist, risk=risk)

print("Graph built")

# ============================================================
# ROBUST NODE MAPPING ✅ (FIX für KeyError)
# ============================================================

graph_nodes = set(G.nodes())

node_ids = nodes.node.values
node_lats = nodes.lat.values
node_lons = nodes.lon.values

def nearest(lat, lon):
    d = haversine_vectorized(lat, lon, node_lats, node_lons)
    sorted_idx = np.argsort(d)

    for idx in sorted_idx:
        candidate = int(node_ids[idx])
        if candidate in graph_nodes:
            return candidate

    raise ValueError("Kein gültiger Knoten gefunden")

points_df["node"] = [
    nearest(r.lat, r.lon) for _, r in points_df.iterrows()
]

# ============================================================
# MULTI OBJECTIVE DIJKSTRA ✅
# ============================================================

def multi_objective_dijkstra(source, lam):

    visited = {}
    heap = [(0, source, 0, 0)]

    while heap:
        total, node, dist, risk = heapq.heappop(heap)

        if node in visited:
            continue

        visited[node] = (dist, risk)

        for nbr in G[node]:
            d = G[node][nbr]["dist"]
            r = G[node][nbr]["risk"]

            new_dist = dist + d
            new_risk = risk + r
            new_total = new_dist + lam * new_risk

            heapq.heappush(heap, (new_total, nbr, new_dist, new_risk))

    return visited

# ============================================================
# OD MATRIX
# ============================================================

records = []

for _, src in points_df.iterrows():

    for profile, lam in PATH_PROFILES.items():

        result_map = multi_objective_dijkstra(src.node, lam)

        for _, dst in points_df.iterrows():

            if src.id == dst.id:
                continue

            if dst.node not in result_map:
                continue

            dist_km, risk = result_map[dst.node]
            time_min = 60 * dist_km / AVG_SPEED_KMH

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "dist_km": dist_km,
                "time_min": time_min,
                "risk": risk
            })

# ============================================================
# SAVE
# ============================================================

od_df = pd.DataFrame(records)
od_df.to_csv(OUTPUT_MATRIX_CSV, index=False)

print("✅ DONE – stabile + realistische OD Matrix erstellt")


Nodes: 2019487 | Edges: 3023347
Graph built
✅ DONE – stabile + realistische OD Matrix erstellt


Mittlere Instanz

In [1]:
# ============================================================
# HAZMAT PREPROCESSING – MITTLERE INSTANZ (~50 KUNDEN)
# KOMPLETTE MATRIX (KEINE LÜCKEN)
# ============================================================

import pandas as pd
import numpy as np
import pickle
import networkx as nx
import heapq

# ============================================================
# SETTINGS
# ============================================================

ACCIDENTS_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_accidents.pkl"
POPULATION_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_population.pkl"
NATURE_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_nature.pkl"

OUTPUT_MATRIX_CSV = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_od_matrix_50.csv"

BOUNDING_BUFFER_DEG = 0.5
AVG_SPEED_KMH = 70.0

PATH_PROFILES = {
    "shortest": 0.0,
    "safest": 4.0
}

alpha, beta, gamma = 0.4, 0.4, 0.2

# ============================================================
# HELPER
# ============================================================

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# ============================================================
# POINTS
# ============================================================

points_df = pd.read_csv(r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Routenplanung_Gefahrguttransport\models\Berlin\customer_50.csv")

# ============================================================
# BOUNDING BOX
# ============================================================

lat_min = points_df.lat.min() - BOUNDING_BUFFER_DEG
lat_max = points_df.lat.max() + BOUNDING_BUFFER_DEG
lon_min = points_df.lon.min() - BOUNDING_BUFFER_DEG
lon_max = points_df.lon.max() + BOUNDING_BUFFER_DEG

# ============================================================
# GRAPH LADEN
# ============================================================

with open(ACCIDENTS_PATH, "rb") as f:
    g = pickle.load(f)

nodes = g["nodes"]
edges = g["edges"]

nodes = nodes[
    (nodes.lat >= lat_min) &
    (nodes.lat <= lat_max) &
    (nodes.lon >= lon_min) &
    (nodes.lon <= lon_max)
].copy()

valid_nodes = set(nodes.node)

edges = edges[
    edges.u.isin(valid_nodes) &
    edges.v.isin(valid_nodes)
].copy()

print(f"Nodes: {len(nodes)} | Edges: {len(edges)}")

# ============================================================
# RISIKO LADEN
# ============================================================

def load_metric(path, col):
    with open(path, "rb") as f:
        g = pickle.load(f)
    df = g["edges"]
    df = df[df.u.isin(valid_nodes) & df.v.isin(valid_nodes)]
    return dict(zip(zip(df.u, df.v), df[col]))

pop = load_metric(POPULATION_PATH, "pop_per_meter")
acc = load_metric(ACCIDENTS_PATH, "score")
nat = load_metric(NATURE_PATH, "dist_to_nature_m")

# ============================================================
# GRAPH BUILD
# ============================================================

node_coords = nodes.set_index("node")[["lat", "lon"]].to_dict("index")

G = nx.DiGraph()

for _, row in edges.iterrows():
    u, v = int(row.u), int(row.v)

    if "length" in row and 0 < row.length < 10000:
        dist = float(row.length) / 1000.0
    else:
        cu, cv = node_coords[u], node_coords[v]
        dist = haversine(cu["lat"], cu["lon"], cv["lat"], cv["lon"]) / 1000.0

    risk = beta * acc.get((u,v), 0) * (
        alpha * pop.get((u,v), 0) + gamma * nat.get((u,v), 0)
    ) * 500

    G.add_edge(u, v, dist=dist, risk=risk)

print("Graph ready")

# ============================================================
# NODE MAPPING
# ============================================================

graph_nodes = set(G.nodes())

node_ids = nodes.node.values
node_lats = nodes.lat.values
node_lons = nodes.lon.values

def nearest(lat, lon):
    d = (node_lats - lat)**2 + (node_lons - lon)**2
    idx_sorted = np.argsort(d)
    for idx in idx_sorted:
        n = int(node_ids[idx])
        if n in graph_nodes:
            return n

points_df["node"] = [
    nearest(r.lat, r.lon) for _, r in points_df.iterrows()
]

# ============================================================
# MULTI DIJKSTRA
# ============================================================

def multi_dijkstra(source, lam):

    visited = {}
    heap = [(0, source, 0, 0)]

    while heap:
        total, node, dist_val, risk_val = heapq.heappop(heap)

        if node in visited:
            continue

        visited[node] = (dist_val, risk_val)

        for nbr in G[node]:
            d = G[node][nbr]["dist"]
            r = G[node][nbr]["risk"]

            heapq.heappush(heap, (
                dist_val + lam * risk_val + d + lam * r,
                nbr,
                dist_val + d,
                risk_val + r
            ))

    return visited

# ============================================================
# OD MATRIX
# ============================================================

records = []

for _, src in points_df.iterrows():

    print(f"Running source: {src.id}")

    for profile, lam in PATH_PROFILES.items():

        res = multi_dijkstra(src.node, lam)

        for _, dst in points_df.iterrows():

            if src.id == dst.id:
                continue

            if dst.node in res:
                dist_km, risk_val = res[dst.node]
            else:
                dist_km = np.nan
                risk_val = np.nan

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "dist_km": dist_km,
                "time_min": 60 * dist_km / AVG_SPEED_KMH if not np.isnan(dist_km) else np.nan,
                "risk": risk_val
            })

# ============================================================
# 🔥 COMPLETE MATRIX FIX
# ============================================================

print("Erzeuge vollständige Matrix...")

od_df_all = pd.DataFrame(records)
all_nodes = points_df["id"].tolist()

complete_records = []

max_dist = od_df_all["dist_km"].max()
max_risk = od_df_all["risk"].max()

fallback_dist = max_dist * 10
fallback_risk = max_risk * 10

for profile in PATH_PROFILES.keys():

    dfp = od_df_all[od_df_all["profile"] == profile]
    existing = set(zip(dfp["from"], dfp["to"]))

    for i in all_nodes:
        for j in all_nodes:

            if i == j:
                continue

            if (i, j) in existing:
                row = dfp[(dfp["from"] == i) & (dfp["to"] == j)].iloc[0]
                complete_records.append(row.to_dict())
            else:
                complete_records.append({
                    "from": i,
                    "to": j,
                    "profile": profile,
                    "dist_km": fallback_dist,
                    "time_min": 60 * fallback_dist / AVG_SPEED_KMH,
                    "risk": fallback_risk
                })

# ============================================================
# SAVE
# ============================================================

od_complete = pd.DataFrame(complete_records)
od_complete.to_csv(OUTPUT_MATRIX_CSV, index=False)

print("✅ Vollständige OD Matrix gespeichert!")
print("Anzahl Einträge:", len(od_complete))

c:\Users\tspol\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\tspol\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Nodes: 1970347 | Edges: 2942840
Graph ready
Running source: DEPOT
Running source: C1
Running source: C2
Running source: C3
Running source: C4
Running source: C5
Running source: C6
Running source: C7
Running source: C8
Running source: C9
Running source: C10
Running source: C11
Running source: C12
Running source: C13
Running source: C14
Running source: C15
Running source: C16
Running source: C17
Running source: C18
Running source: C19
Running source: C20
Running source: C21
Running source: C22
Running source: C23
Running source: C24
Running source: C25
Running source: C26
Running source: C27
Running source: C28
Running source: C29
Running source: C30
Running source: C31
Running source: C32
Running source: C33
Running source: C34
Running source: C35
Running source: C36
Running source: C37
Running source: C38
Running source: C39
Running source: C40
Running source: C41
Running source: C42
Running source: C43
Running source: C44
Running source: C45
Running source: C46
Running source: C47
Run

In [2]:
OUTPUT_MATRIX_CSV = r"C:\Users\tspol\Sciebo\OperationsResearch\od_matrix_50.csv"

In [3]:
pd.DataFrame(records).to_csv(OUTPUT_MATRIX_CSV, index=False)

In [ ]:
# ============================================================
# HAZMAT PREPROCESSING – MITTLERE INSTANZ (~50 KUNDEN)
# ============================================================

import pandas as pd
import numpy as np
import pickle
import networkx as nx
import heapq

# ============================================================
# SETTINGS
# ============================================================

ACCIDENTS_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_accidents.pkl"
POPULATION_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_population.pkl"
NATURE_PATH = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_graph_with_nature.pkl"

#OUTPUT_MATRIX_CSV = r"C:\Users\tspol\Sciebo\OperationsResearch\germany_od_matrix_50.csv"

BOUNDING_BUFFER_DEG = 0.5   # <<< kleiner!
AVG_SPEED_KMH = 70.0

PATH_PROFILES = {
    "shortest": 0.0,
    "safest": 4.0
}

alpha, beta, gamma = 0.4, 0.4, 0.2

# ============================================================
# HELPER
# ============================================================

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# ============================================================
# POINTS (HIER DEINE 50 KUNDEN!)
# ============================================================

points_df = pd.read_csv(r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Routenplanung_Gefahrguttransport\models\Berlin\customer_50.csv")  # empfehlenswert

# erwartet:
# id, lat, lon

# ============================================================
# BOUNDING BOX
# ============================================================

lat_min = points_df.lat.min() - BOUNDING_BUFFER_DEG
lat_max = points_df.lat.max() + BOUNDING_BUFFER_DEG
lon_min = points_df.lon.min() - BOUNDING_BUFFER_DEG
lon_max = points_df.lon.max() + BOUNDING_BUFFER_DEG

# ============================================================
# GRAPH LADEN
# ============================================================

with open(ACCIDENTS_PATH, "rb") as f:
    g = pickle.load(f)

nodes = g["nodes"]
edges = g["edges"]

nodes = nodes[
    (nodes.lat >= lat_min) &
    (nodes.lat <= lat_max) &
    (nodes.lon >= lon_min) &
    (nodes.lon <= lon_max)
].copy()

valid_nodes = set(nodes.node)

edges = edges[
    edges.u.isin(valid_nodes) &
    edges.v.isin(valid_nodes)
].copy()

print(f"Nodes: {len(nodes)} | Edges: {len(edges)}")

# ============================================================
# STRASSEN FILTER (SEHR EMPFOHLEN)
# ============================================================

if "highway" in edges.columns:
    edges = edges[edges["highway"].isin([
        "motorway", "trunk", "primary", "secondary"
    ])]

# ============================================================
# RISIKO LADEN
# ============================================================

def load_metric(path, col):
    with open(path, "rb") as f:
        g = pickle.load(f)
    df = g["edges"]
    df = df[df.u.isin(valid_nodes) & df.v.isin(valid_nodes)]
    return dict(zip(zip(df.u, df.v), df[col]))

pop = load_metric(POPULATION_PATH, "pop_per_meter")
acc = load_metric(ACCIDENTS_PATH, "score")
nat = load_metric(NATURE_PATH, "dist_to_nature_m")

# ============================================================
# GRAPH BUILD
# ============================================================

node_coords = nodes.set_index("node")[["lat", "lon"]].to_dict("index")

G = nx.DiGraph()

for _, row in edges.iterrows():
    u, v = int(row.u), int(row.v)

  
    # reale Distanz aus Geometrie
    if "length" in row and 0 < row.length < 10000:
        base_dist = float(row.length) / 1000.0
    else:
        cu = node_coords[u]
        cv = node_coords[v]
        base_dist = haversine(cu["lat"], cu["lon"], cv["lat"], cv["lon"]) / 1000.0

    # 👉 GLOBALER KALIBRIERUNGSFAKTOR (GANZ WICHTIG!)
    DIST_CORRECTION = 0.65
    base_dist *= DIST_CORRECTION


    risk = beta * acc.get((u,v), 0) * (
        alpha * pop.get((u,v), 0) + gamma * nat.get((u,v), 0)
    ) * 500

    G.add_edge(u, v, dist=dist, risk=risk)

print("Graph ready")

# ============================================================
# ROBUST MAPPING
# ============================================================

graph_nodes = set(G.nodes())

node_ids = nodes.node.values
node_lats = nodes.lat.values
node_lons = nodes.lon.values

def nearest(lat, lon):
    d = (node_lats - lat)**2 + (node_lons - lon)**2
    idx_sorted = np.argsort(d)
    for idx in idx_sorted:
        n = int(node_ids[idx])
        if n in graph_nodes:
            return n

points_df["node"] = [
    nearest(r.lat, r.lon) for _, r in points_df.iterrows()
]

# ============================================================
# FAST MULTI DIJKSTRA
# ============================================================

def multi_dijkstra(source, lam):

    visited = {}
    heap = [(0, source, 0, 0)]

    while heap:
        total, node, dist, risk = heapq.heappop(heap)

        if node in visited:
            continue

        visited[node] = (dist, risk)

        for nbr in G[node]:
            d = G[node][nbr]["dist"]
            r = G[node][nbr]["risk"]

            nd = dist + d
            nr = risk + r

            heapq.heappush(heap, (nd + lam * nr, nbr, nd, nr))

    return visited

# ============================================================
# OD MATRIX
# ============================================================

records = []

for _, src in points_df.iterrows():

    print(f"Running source: {src.id}")

    for profile, lam in PATH_PROFILES.items():

        res = multi_dijkstra(src.node, lam)

        for _, dst in points_df.iterrows():

            if src.id == dst.id:
                continue

            if dst.node not in res:
                continue

            dist_km, risk = res[dst.node]

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "dist_km": dist_km,
                "time_min": 60 * dist_km / AVG_SPEED_KMH,
                "risk": risk
            })

# ============================================================
# SAVE
# ============================================================

#pd.DataFrame(records).to_csv("od_matrix_50.csv", index=False)

print("✅ Fertig – mittlere Instanz berechnet")

Nodes: 1970347 | Edges: 2942840


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000002D648866E70>>
Traceback (most recent call last):
  File "C:\Users\tspol\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py", line 790, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tspol\anaconda3\Lib\threading.py", line 1533, in enumerate
    def enumerate():
    
KeyboardInterrupt: 


In [ ]:
import folium
import networkx as nx

# ============================================================
# EINZELROUTE VISUALISIERUNG (DEPOT -> C2)
# ============================================================

START_ID = "DEPOT"
END_ID = "C2"

# Mapping aus deinem points_df
point_to_node = dict(zip(points_df["id"], points_df["node"]))

start_node = point_to_node[START_ID]
end_node = point_to_node[END_ID]

# ============================================================
# GEWICHTSFUNKTIONEN
# ============================================================

def weight_shortest(u, v, d):
    return d["dist"]

def weight_safest(u, v, d):
    return d["dist"] + 4.0 * d["risk"]

# ============================================================
# PFADE BERECHNEN
# ============================================================

print("Berechne shortest path...")
path_shortest = nx.shortest_path(
    G,
    source=start_node,
    target=end_node,
    weight=weight_shortest
)

print("Berechne safest path...")
path_safest = nx.shortest_path(
    G,
    source=start_node,
    target=end_node,
    weight=weight_safest
)

# ============================================================
# KOORDINATEN EXTRAHIEREN
# ============================================================

def extract_coords(path):
    coords = []
    for n in path:
        coord = node_coords[n]
        coords.append((coord["lat"], coord["lon"]))
    return coords

coords_shortest = extract_coords(path_shortest)
coords_safest = extract_coords(path_safest)

# ============================================================
# KARTE ERSTELLEN
# ============================================================

# Mittelpunkt der Karte
center_lat = coords_shortest[0][0]
center_lon = coords_shortest[0][1]

m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

# ============================================================
# ROUTEN EINZEICHNEN
# ============================================================

# Shortest = Blau
folium.PolyLine(
    coords_shortest,
    color="blue",
    weight=4,
    tooltip="Shortest Path"
).add_to(m)

# Safest = Rot
folium.PolyLine(
    coords_safest,
    color="red",
    weight=4,
    tooltip="Safest Path"
).add_to(m)

# ============================================================
# MARKER
# ============================================================

# Start
folium.Marker(
    location=coords_shortest[0],
    popup=f"Start: {START_ID}",
    icon=folium.Icon(color="green")
).add_to(m)

# Ziel
folium.Marker(
    location=coords_shortest[-1],
    popup=f"Ziel: {END_ID}",
    icon=folium.Icon(color="red")
).add_to(m)

# ============================================================
# SPEICHERN
# ============================================================

output_file = r"C:\Users\tspol\Sciebo\OperationsResearch\route_DEPOT_C2.html"
m.save(output_file)

print("✅ Karte gespeichert unter:")
print(output_file)

Berechne shortest path...
